# 02 Feature engineering
Shows the leakage check on the real data: changing every line on or after the cutoff must not change any feature.

**Note:** this notebook is illustrative and calls `build_features` without the training-fitted reference statistics, so product medians, StockCode frequencies and engagement ranks are computed from the whole customer population here. It is a leakage demonstration only. The production pipeline (`python -m src.pipeline`) fits these statistics on training customers only and persists them in `reference_stats.joblib`.

In [1]:
import sys
import pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.utils.config import load_config, rpath
cfg = load_config()
from src.features.build import build_features, FEATURE_COLUMNS

In [2]:
tx = pd.read_parquet(rpath(cfg,'interim_transactions'))
cut = cfg['features']['cutoff_date']
feats, tg = build_features(tx, cut, cfg['features']['churn_horizon_days'])
print(feats.shape, 'cutoff', cut)
feats[FEATURE_COLUMNS].head()

(5248, 39) cutoff 2011-09-09


,recency_days,orders_last_30d,orders_last_90d,orders_last_180d,spend_last_30d,spend_last_90d,spend_last_180d,recent_share_90d,days_since_first_purchase,frequency,...,discount_sensitivity,product_popularity_mean,niche_product_share,country_united_kingdom,country_eire,country_germany,country_france,country_other,clv_historical,engagement_score
customer_id,,,,,,,,,,,,,,,,,,,,,
12346,437,0.0,0.0,0.0,0.0,0.00,0.00,0.000000,555,2,...,0.000000,5.426624,0.0,1,0,0,0,0,-77014.24,20.804751
12347,37,0.0,1.0,3.0,0.0,584.91,1603.68,0.166667,312,6,...,0.036585,5.102716,0.0,0,0,0,0,1,3402.39,79.573171
12348,156,0.0,0.0,1.0,0.0,0.00,327.00,0.000000,346,4,...,0.090909,5.616110,0.0,0,0,0,0,1,1388.40,59.584604
12349,315,0.0,0.0,0.0,0.0,0.00,0.00,0.000000,497,2,...,0.040000,5.408882,0.0,0,0,0,0,1,2196.99,47.551448
12350,218,0.0,0.0,0.0,0.0,0.00,0.00,0.000000,218,1,...,0.000000,5.077581,0.0,0,0,0,0,1,294.40,26.730818


In [3]:
mut = tx.copy(); fut = mut.invoice_date >= pd.Timestamp(cut)
mut.loc[fut,'price'] = 999.0; mut.loc[fut,'quantity'] = 50; mut.loc[fut,'line_total'] = 999.0*50
f2,_ = build_features(mut, cut, cfg['features']['churn_horizon_days'])
pd.testing.assert_frame_equal(feats, f2)
print('leakage check passed: features identical after rewriting', int(fut.sum()), 'future lines')

leakage check passed: features identical after rewriting 162554 future lines


In [4]:
feats[FEATURE_COLUMNS].isna().sum().sum(), feats[FEATURE_COLUMNS].describe().T.round(2).head(12)

(np.int64(0),
                             count     mean       std   min     25%     50%  \
 recency_days               5248.0   205.86    173.95  0.00   49.00  163.00   
 orders_last_30d            5248.0     0.25      0.71  0.00    0.00    0.00   
 orders_last_90d            5248.0     0.74      1.77  0.00    0.00    0.00   
 orders_last_180d           5248.0     1.51      3.35  0.00    0.00    1.00   
 spend_last_30d             5248.0   118.20    777.87  0.00    0.00    0.00   
 spend_last_90d             5248.0   341.47   1923.19  0.00    0.00    0.00   
 spend_last_180d            5248.0   669.95   3311.21  0.00    0.00  107.01   
 recent_share_90d           5248.0     0.13      0.26  0.00    0.00    0.00   
 days_since_first_purchase  5248.0   429.92    180.37  0.00  309.00  472.00   
 frequency                  5248.0     5.70     11.18  1.00    1.00    3.00   
 monetary_total             5248.0  2537.47  11882.78  1.55  316.46  775.16   
 monetary_avg               5248.0   3

In [5]:
print(pd.concat([feats.engagement_score, tg.churn],axis=1).groupby(pd.qcut(feats.engagement_score,5)).churn.mean().round(3))

engagement_score
(5.252, 24.425]     0.870
(24.425, 41.066]    0.741
(41.066, 57.065]    0.603
(57.065, 75.142]    0.446
(75.142, 99.74]     0.170
Name: churn, dtype: float64


/tmp/ipykernel_24813/2931756730.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(pd.concat([feats.engagement_score, tg.churn],axis=1).groupby(pd.qcut(feats.engagement_score,5)).churn.mean().round(3))
